# Lekcija 11 - Protokol modelnega konteksta (MCP)

The **Model Context Protocol (MCP)** je odprt standard, ki agentom omogoča dinamično odkrivanje in uporabo orodij, virov in podatkovnih virov med izvajanjem. Namesto da bi orodja trdo kodirali v agenta, MCP agentom omogoča povezovanje z zunanjimi strežniki, ki na zahtevo razkrivajo svoje zmožnosti.

V tej lekciji se boste naučili:
- Kaj je MCP in zakaj je pomemben za agentske sisteme
- Kako deluje odjemalsko-strežniška arhitektura MCP
- Kako zgraditi agente, ki uporabljajo odkrivanje orodij v slogu MCP


## Nastavitev

**Predpogoji:**
- Projekt Azure AI Foundry z razporejenim modelom
- Za preverjanje pristnosti zaženite `az login`


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Kaj je Model Context Protocol (MCP)?

MCP določa standarden način, kako AI agenti odkrijejo in komunicirajo z zunanjimi orodji in viri podatkov:

- **MCP Server**: Izpostavlja orodja, vire in pozive prek standardnega protokola
- **MCP Client**: Izvedbeno okolje agenta, ki se poveže s strežniki in odkrije razpoložljive zmogljivosti
- **Dynamic Discovery**: Agenti ne potrebujejo trdo kodiranih orodij — odkrijejo, kaj je na voljo med izvajanjem

To je zmogljivo pri gradnji razširljivih agentnih sistemov, kjer je mogoče dodajati nove zmogljivosti brez spreminjanja kode agenta.


## Kako MCP deluje

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (MCP client) se poveže z MCP server
2. Strežnik odgovori s seznamom razpoložljivih orodij in njihovih shem
3. Agent lahko nato pokliče katerokoli odkrito orodje med svojim sklepanjem
4. Rezultati se vrnejo po istem protokolu


## Simuliranje odkrivanja orodij MCP

Ker pravi strežnik MCP zahteva delujoč strežniški proces, bomo vzorec prikazali z uporabo funkcij `@tool`, ki simulirajo, kaj bi storitev za nastanitve, povezana z MCP, zagotovila.

V produkciji bi bila ta orodja dinamično odkrita s strežnika MCP, namesto da bi bila definirana lokalno.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Gradnja agenta z orodji v slogu MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP v produkcijskem okolju

V produkcijskem okolju MCP omogoča zmogljive vzorce:

- **Dinamično odkrivanje orodij**: Agenti se povežejo s strežniki MCP in med izvajanjem odkrijejo orodja
- **Ločena arhitektura**: Ponudniki orodij se lahko posodabljajo neodvisno od agenta
- **Deljenje med organizacijami**: Ekipe lahko preko strežnikov MCP izpostavijo zmogljivosti, ki jih lahko uporabi kateri koli agent
- **Podpora Microsoft Agent Frameworka**: MAF ima vgrajeno podporo MCP odjemalca prek integracije `mcp`

Za uporabo dejanskega strežnika MCP z MAF bi se povezali preko `hosted_mcp_tool()` ali integracije MCP odjemalca.

**Več informacij:**
- [Specifikacija MCP](https://modelcontextprotocol.io/)
- [Podpora MCP v Microsoft Agent Frameworku](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Povzetek

V tej lekciji ste se naučili:
- **MCP** je odprt standard za dinamično odkrivanje orodij med agenti in ponudniki orodij
- Arhitektura **odjemalec-strežnik** agentom omogoča odkrivanje zmogljivosti med izvajanjem
- MCP omogoča **razširljive, neodvisne sisteme agentov**, kjer je mogoče dodajati orodja brez sprememb kode
- Microsoft Agent Framework zagotavlja **vgrajeno podporo za MCP** za uporabo v produkciji


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Omejitev odgovornosti:
Ta dokument je bil preveden z uporabo storitve za prevajanje z umetno inteligenco [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, upoštevajte, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v izvorni različici naj velja za avtoritativni vir. Za kritične informacije priporočamo strokovni prevod, opravljen s strani človeka. Za morebitne nesporazume ali napačne razlage, ki izhajajo iz uporabe tega prevoda, ne odgovarjamo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
